# MLX Graph Models - CrossValidation (MLX)

The following tutorial will demo how to run Cross-Validation scripts for the graph models. These methods condense parameters needed when generating graph models as demoed in [mlx_model.ipynb]("mlx_model.ipynb").

## Import Modules
Import the modules needed to build the model and run cross-validation.

In [1]:
import pandas as pd
from mminer_graphs.mlx_models import GCNNet, CrossVal

## Dataset Prep
This tutorial will use the bace dataset taken from [MoleculeNet](https://moleculenet.org). This dataset contains additional columns with train/test split labels that were randomly assigned. The labels for these columns are CVx where x is the fold number. The relavent information needed for this model will be extracted as follows:

In [2]:
data = pd.read_csv('../dataset/bace.csv')
cols = ['CID', 'smiles', 'Class', 'CV0', 'CV1', 'CV2', 'CV3', 'CV4']
dataset = data[cols]
dataset.head()

,CID,smiles,Class,CV0,CV1,CV2,CV3,CV4
0,BACE_1,O1CC[C@@H](NC(=O)[C@@H](Cc2cc3cc(ccc3nc2N)-c2c...,1,train,train,train,train,train
1,BACE_2,Fc1cc(cc(F)c1)C[C@H](NC(=O)[C@@H](N1CC[C@](NC(...,1,train,train,train,train,train
2,BACE_3,S1(=O)(=O)N(c2cc(cc3c2n(cc3CC)CC1)C(=O)N[C@H](...,1,train,train,train,train,test
3,BACE_4,S1(=O)(=O)C[C@@H](Cc2cc(O[C@H](COCC)C(F)(F)F)c...,1,train,train,train,train,test
4,BACE_5,S1(=O)(=O)N(c2cc(cc3c2n(cc3CC)CC1)C(=O)N[C@H](...,1,train,train,train,train,train


### Optional
As mentioned in the [mlx_model.ipynb]("mlx_model.ipynb") tutorial, additional features can be added to the model. An example of generating this is as follows:

In [3]:
# extra features
extra_cols = ["pIC50", "MW", "AlogP", "HBA", "HBD", "RB", "HeavyAtomCount", "ChiralCenterCount",
              "ChiralCenterCountAllPossible", "RingCount"]
extra_data = data[extra_cols]
extra_list = extra_data.values.tolist()

## Build Model
The GNN is set up similar to previous notebooks.

In [4]:
model = GCNNet(node_dim=142, hidden_dim=64, graph_layers=2, mlp_layers=2, reduce_graph=False, mlp_dropout=0, num_classes=2, add_feats=extra_list)
model

GCNNet(
  (gcn_conv.0): GCNConv(
    (linear): Linear(input_dims=142, output_dims=64, bias=True)
  )
  (gcn_conv.1): GCNConv(
    (linear): Linear(input_dims=64, output_dims=64, bias=True)
  )
  (normalization.0): BatchNorm(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (normalization.1): BatchNorm(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (mlp.0): Sequential(
    (layers.0): Linear(input_dims=74, output_dims=37, bias=True)
    (layers.1): BatchNorm(37, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (layers.2): ReLU()
    (layers.3): Dropout(p=0)
  )
  (final_mlp): Linear(input_dims=37, output_dims=2, bias=True)
)

## Cross-Validation
Instantiating the **CrossVal()** class contains three parameters. The model and dataset parameters are optional and, if not given when instantiating **CrossVal()**, can also be given under the method **get_cv_scores()**. Here, the model and dataset is given to the **CrossVal()** class. Additionally, there is an **estop** parameter. This is the early stopping parameter and requires an int that will determine the number of epochs a given CV fold will run before stopping. Additionally, users can use the mol-miner-graphs **EarlyStopping()** class to create a custom monitor and mode to trigger early stopping.

In [5]:
cv = CrossVal(model=model, dataset=dataset, estop=75)

## Get CV Scores
The **get_cv_scores()** method contains many parameters and should mirror what is needed to run the models individually as seen in [mlx_model.ipynb]("mlx_model.ipynb"). However, it comes with additional parameters unique for its function. The pertinent parameters are shown in this tutorial.

By default, this method will convert the smiles strings from the input dataset using **AtomGraph**. Additional features, if needed, must be given to this method as a list. Finally a progress bar and verbosity for each CV Round can be Turned on or off depending on user needs.

In [6]:
metrics = cv.get_cv_scores(graph_type="AtomGraph", labels="CV", cv=5, smi_col='smiles', activity_col='Class',
                           epochs=100, batch_size=1024, add_feats=extra_list, verbose=False, pbar=True)

CV Round:   0%|          | 0/5 [00:00<?, ?it/s]

Model running on Apple M2 Pro


CV Round:  20%|██        | 1/5 [00:36<02:26, 36.59s/it]

Best Training Loss: 0.019
Best Test Loss: 0.153
Best Training Accuracy: 0.996
Best Test Accuracy: 0.928
Model running on Apple M2 Pro


CV Round:  40%|████      | 2/5 [01:10<01:45, 35.27s/it]

Best Training Loss: 0.027
Best Test Loss: 0.09
Best Training Accuracy: 0.996
Best Test Accuracy: 0.971
Model running on Apple M2 Pro


... (2 outputs left out)

CV Round:  80%|████████  | 4/5 [02:19<00:34, 34.60s/it]

Best Training Loss: 0.026
Best Test Loss: 0.036
Best Training Accuracy: 0.996
Best Test Accuracy: 0.993
Model running on Apple M2 Pro


CV Round: 100%|██████████| 5/5 [02:54<00:00, 34.90s/it]

Best Training Loss: 0.025
Best Test Loss: 0.09
Best Training Accuracy: 0.996
Best Test Accuracy: 0.942


## CV Metrics
The output is given as a pd.DataFrame. This can be saved as a .csv file or, depending on the user needs, modified for further analysis.

In [7]:
metrics

,Accuracy,Precision,Recall,F1,MCC,ROC_AUC,Loss
fold,,,,,,,
0,0.927536,0.931507,0.931507,0.931507,0.854584,0.989673,0.153284
1,0.971014,0.982143,0.948276,0.964912,0.940654,0.995259,0.090133
2,0.978261,1.000000,0.948276,0.973451,0.956034,0.998276,0.065871
3,0.992754,0.984375,1.000000,0.992126,0.985520,0.999365,0.035890
4,0.942029,0.921053,0.972222,0.945946,0.885046,0.994529,0.090375


In [8]:
cv.summary

,Accuracy,Precision,Recall,F1,MCC,ROC_AUC,Loss
mean,0.962,0.964,0.960,0.962,0.924,0.995,0.087
std,0.027,0.035,0.027,0.024,0.053,0.004,0.043
